In [ ]:
import os
from math import exp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import statsmodels.api as sm
import plotly.graph_objects as go


plt.ion() # matplotlib interactively
import os
# portable repo-root resolution (walk up to the .git / .here marker)
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, '.git')) or os.path.isfile(os.path.join(ROOT, '.here'))) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)


In [ ]:
index = pd.read_csv("./0_raw/climate_finance/glcf_country_index.csv")
glcf = pd.read_csv("./0_raw/climate_finance/glcf_raw_2023_v2.csv")
pop = pd.read_csv("./0_raw/climate_finance/population_wdi_processed.csv")
pop = pop[pop["year"] == 2021][['ISO', 'pop']]

In [ ]:
glcf['instrument_cpi0'] = glcf['instrument_cpi'].replace({
    "Balance sheet financing (debt portion)": "Debt",
    "Balance sheet financing (equity portion)": "Equity",
    "Project-level market rate debt": "Debt",
    'Project-level equity': "Equity",
    'Low-cost project debt': "Debt",
    'Grant': "Grant",
    "Unknown": "Unknown"
})
glcf['region_destination_cpi0'] = glcf['region_destination_cpi'].apply(
    lambda x: "Rest of the World" if x not in ['East Asia and Pacific', "US & Canada", "Western Europe"] else x
)
glcf['instrument_combined'] = (
    glcf['domestic_international'] + ": " + glcf['instrument_cpi0'])

In [ ]:
region_map = {
    "Central Asia and Eastern Europe": 0,
    "East Asia and Pacific": 1,
    "Latin America & Caribbean": 2,
    "Middle East and North Africa": 3,
    "Other Oceania": 4,
    "South Asia": 5,
    "Sub-Saharan Africa": 6,
    "US & Canada": 7,
    "Western Europe": 8,
    "Transregional": 9
}

add_nodes = {
    "Domestic": 10,
    "International": 11,
    "Adaptation": 12,
    "Mitigation": 13,
    "Multiple Objectives": 14
}


In [ ]:
glcf1 = glcf.groupby(['domestic_international', 'region_destination_cpi'], as_index=False).agg(
    total=('value_USDm', lambda x: sum(x / 2)))
glcf2 = glcf.groupby(['region_destination_cpi', 'use_cpi'], as_index=False).agg(
    total=('value_USDm', lambda x: sum(x / 2)))

In [ ]:
# sankey1: Domestic/Intl to region
sankey1 = (
    glcf1[glcf1['region_destination_cpi'] != "Unknown"]
    .assign(
        source=lambda d: d['domestic_international'].map({'Domestic': 10, 'International': 11}),
        target=lambda d: d['region_destination_cpi'].map(region_map),
        value=lambda d: d['total']
    )[["source", "target", "value"]]
)

# sankey2: Region to Use
sankey2 = (
    glcf2[(glcf2['region_destination_cpi'] != "Unknown") & 
                (glcf2['use_cpi'] != "Unknown")]
    .assign(
        source=lambda d: d['region_destination_cpi'].map(region_map),
        target=lambda d: d['use_cpi'].map({
            "Adaptation": 12,
            "Mitigation": 13,
            "Multiple Objectives": 14
        }),
        value=lambda d: d['total']
    )[["source", "target", "value"]]
)

links_df = pd.concat([sankey1, sankey2], ignore_index=True)


In [ ]:
# Combine all unique node names and preserve ordering
node_labels = [
    "Central Asia and Eastern Europe", "East Asia and Pacific", "Latin America & Caribbean",
    "Middle East and North Africa", "Other Oceania", "South Asia", "Sub-Saharan Africa",
    "US & Canada", "Western Europe", "Transregional",
    "Domestic", "International",
    "Adaptation", "Mitigation", "Multiple Objectives"
]
colors = [
    "#FF6969",  # regions
    "steelblue", "#FF6969",  # Domestic, International
    "#FF6600", "#E6C767", "#898121"  # Adaptation, Mitigation, Multiple
]


In [ ]:
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=50,
        line=dict(color="white", width=0.5),
        label=node_labels,
        color=colors
    ),
    link=dict(
        source=links_df['source'],
        target=links_df['target'],
        value=links_df['value']
    )
)])

fig.update_layout(title_text="Climate Finance", font_size=13)
fig.show()


# by instrument

In [ ]:
# First flow: instrument_cpi → region_destination_cpi
glcf1 = (
    glcf[glcf['region_destination_cpi'] != 'Unknown']  # filter unknowns
    .groupby(['instrument_cpi0', 'region_destination_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf1['total'] = glcf1['value_USDm'] / 2
glcf1 = glcf1.drop(columns='value_USDm')

# Second flow: region_destination_cpi → use_cpi
glcf2 = (
    glcf[(glcf['region_destination_cpi'] != 'Unknown') & (glcf['use_cpi'] != 'Unknown')]
    .groupby(['region_destination_cpi', 'use_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf2['total'] = glcf2['value_USDm'] / 2
glcf2 = glcf2.drop(columns='value_USDm')

# Unique labels from both sides of links
source_labels_1 = glcf1['instrument_cpi0'].unique().tolist()
mid_labels = glcf1['region_destination_cpi'].unique().tolist()
target_labels = glcf2['use_cpi'].unique().tolist()

# Combine and assign index
all_labels = source_labels_1 + mid_labels + target_labels
label_to_index = {label: i for i, label in enumerate(all_labels)}


In [ ]:
# First set of links: instrument → region
links1 = glcf1.copy()
links1['source'] = links1['instrument_cpi0'].map(label_to_index)
links1['target'] = links1['region_destination_cpi'].map(label_to_index)

# Second set of links: region → use
links2 = glcf2.copy()
links2['source'] = links2['region_destination_cpi'].map(label_to_index)
links2['target'] = links2['use_cpi'].map(label_to_index)

# Combine
all_links = pd.concat([links1[['source', 'target', 'total']], links2[['source', 'target', 'total']]], ignore_index=True)


In [ ]:
debt_index = label_to_index.get("Debt")

# Assign color to each link: blue for Debt, light gray otherwise
link_colors = [
    "rgba(70,130,180,0.6)" if src == debt_index else "rgba(192,192,192,0.4)"
    for src in all_links['source']
]

In [ ]:
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=35,
        label=all_labels,
        color="lightgray"
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['total'],
        color=link_colors
    )
)])

# Set narrower figure dimensions
fig.update_layout(
    title_text="Instrument → Region → Use Sankey",
    font_size=13,
    width=1000,
    height=500
)

fig.show()


# source-instr-dest-use

In [ ]:
# 0. FILTER DATA # 
glcf_filtered = glcf[
    (glcf['instrument_cpi'] != 'Unknown') 
].copy()

# 1. ASSIGN LAYERS # 
# Layer 1: Combined Source → Region
glcf1 = (
    glcf_filtered
    .groupby(['instrument_combined', 'region_destination_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf1['total'] = glcf1['value_USDm'] / 2
glcf1 = glcf1.drop(columns='value_USDm')

# Layer 2: Region → Use (can still use original filter if desired)
glcf2 = (
    glcf_filtered[glcf_filtered['use_cpi'] != 'Unknown']
    .groupby(['region_destination_cpi', 'use_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf2['total'] = glcf2['value_USDm'] / 2
glcf2 = glcf2.drop(columns='value_USDm')

# 2. RETRIEVE AND INDEX LABELS # 
source_labels = glcf1['instrument_combined'].unique().tolist()
mid_labels = glcf1['region_destination_cpi'].unique().tolist()
target_labels = glcf2['use_cpi'].unique().tolist()

all_labels = source_labels + mid_labels + target_labels
label_to_index = {label: i for i, label in enumerate(all_labels)}

# Combined Source → Region
links1 = glcf1.copy()
links1['source'] = links1['instrument_combined'].map(label_to_index)
links1['target'] = links1['region_destination_cpi'].map(label_to_index)

# Region → Use
links2 = glcf2.copy()
links2['source'] = links2['region_destination_cpi'].map(label_to_index)
links2['target'] = links2['use_cpi'].map(label_to_index)

all_links = pd.concat([links1[['source', 'target', 'total']], links2[['source', 'target', 'total']]], ignore_index=True)

# 3. COLOR ASSIGNMENT # 
highlight_index = label_to_index.get("International: Debt")
link_colors = [
    "rgba(70,130,180,0.6)" if src == highlight_index else "rgba(192,192,192,0.4)"
    for src in all_links['source']
]

In [ ]:
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=10,
        thickness=15,
        label=all_labels,
        color="lightgray"
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['total'],
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Combined Source → Region → Use Sankey",
    font_size=13,
    width=800,
    height=600
)

fig.show()


# inst-dom/intl-dest-use

In [ ]:
# Layer 0: Institution type → Combined instrument
glcf0 = (
    glcf_filtered
    .groupby(['institution_type_layer1', 'instrument_combined'], as_index=False)['value_USDm']
    .sum()
)
glcf0['total'] = glcf0['value_USDm'] / 2
glcf0 = glcf0.drop(columns='value_USDm')

# Layer 1: Combined instrument → Region
glcf1 = (
    glcf_filtered
    .groupby(['instrument_combined', 'region_destination_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf1['total'] = glcf1['value_USDm'] / 2
glcf1 = glcf1.drop(columns='value_USDm')

# Layer 2: Region → Use
glcf2 = (
    glcf_filtered
    .groupby(['region_destination_cpi', 'use_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf2['total'] = glcf2['value_USDm'] / 2
glcf2 = glcf2.drop(columns='value_USDm')

# 2. RETRIEVE AND INDEX LABELS #
level0_labels = glcf0['institution_type_layer1'].unique().tolist()
level1_labels = glcf0['instrument_combined'].unique().tolist()
level2_labels = glcf1['region_destination_cpi'].unique().tolist()
level3_labels = glcf2['use_cpi'].unique().tolist()

all_labels = level0_labels + level1_labels + level2_labels + level3_labels
label_to_index = {label: i for i, label in enumerate(all_labels)}

# 3. COMBINE LINKS #
# institution_type → instrument
links0 = glcf0.copy()
links0['source'] = links0['institution_type_layer1'].map(label_to_index)
links0['target'] = links0['instrument_combined'].map(label_to_index)

# instrument → region
links1 = glcf1.copy()
links1['source'] = links1['instrument_combined'].map(label_to_index)
links1['target'] = links1['region_destination_cpi'].map(label_to_index)

# region → use
links2 = glcf2.copy()
links2['source'] = links2['region_destination_cpi'].map(label_to_index)
links2['target'] = links2['use_cpi'].map(label_to_index)

# Final link dataframe
all_links = pd.concat([
    links0[['source', 'target', 'total']],
    links1[['source', 'target', 'total']],
    links2[['source', 'target', 'total']]
], ignore_index=True)

In [ ]:
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=10,
        thickness=15,
        label=all_labels,
        color="lightgray"
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['total'],
        color="rgba(192,192,192,0.4)"
    )
)])

fig.update_layout(
    title_text="Institution Type → Instrument → Region → Use Sankey",
    font_size=13,
    width=900,
    height=600
)

fig.show()


# inst-dom/intl-dest-use v2

In [ ]:
glcf['institution_type_layer2'].unique()

In [ ]:
glcf['institution_type0'] = np.select(
    [
        (glcf['institution_type_layer1'] == 'Private'),
        (glcf['institution_type_layer2'] == 'Bilateral DFI'),
        (glcf['institution_type_layer2'] == 'Multilateral DFI'),
        ~(glcf['institution_type_layer2'].isin(['Bilateral DFI', 'Multilateral DFI']))
    ],
    [
        'Private',
        'Public_Bilat',
        'Public_multilateral',
        'Public_other'
    ],
    default='Unknown'
)

In [ ]:
glcf_filtered['instrument_combined'].unique()

In [ ]:
funding_type_map = {
    'Domestic: Grant': "Domestic",
    'International: Grant': "International",
    'Domestic: Debt': "Domestic",
    'International: Debt': "International",
    'Domestic: Equity': "Domestic",
    'International: Equity': "International"
}


In [ ]:
# 0. FILTER DATA # 
glcf_filtered = glcf[
    (glcf['instrument_cpi'] != 'Unknown') &
    (glcf['use_cpi'] != 'Unknown') 
].copy()

glcf_filtered['funding_type'] = glcf_filtered['instrument_combined'].map(funding_type_map)

# 1. ASSIGN LAYERS # 
# Layer 0: Institution type → Combined instrument
glcf0 = (
    glcf_filtered
    .groupby(['institution_type0', 'instrument_combined'], as_index=False)['value_USDm']
    .sum()
)
glcf0['total'] = glcf0['value_USDm'] / 2
glcf0 = glcf0.drop(columns='value_USDm')

# Layer 1: Combined instrument → Region
glcf1 = (
    glcf_filtered
    .groupby(['instrument_combined', 'region_destination_cpi0'], as_index=False)['value_USDm']
    .sum()
)
glcf1['total'] = glcf1['value_USDm'] / 2
glcf1 = glcf1.drop(columns='value_USDm')

# Layer 2: Region → Use
glcf2 = (
    glcf_filtered
    .groupby(['region_destination_cpi0', 'use_cpi'], as_index=False)['value_USDm']
    .sum()
)
glcf2['total'] = glcf2['value_USDm'] / 2
glcf2 = glcf2.drop(columns='value_USDm')

# 2. RETRIEVE AND INDEX LABELS #
funding_order = ["Domestic", "International"]
instrument_ordered = []
for group in funding_order:
    instruments_in_group = glcf_filtered[
        glcf_filtered['funding_type'] == group
    ]['instrument_combined'].unique()
    instrument_ordered.extend(sorted(instruments_in_group))  # within-group alphabetical

level0_labels = glcf0['institution_type0'].unique().tolist()
level1_labels = instrument_ordered  # use custom-ordered Layer 1
level2_labels = glcf1['region_destination_cpi0'].unique().tolist()
level3_labels = glcf2['use_cpi'].unique().tolist()

all_labels = level0_labels + level1_labels + level2_labels + level3_labels
label_to_index = {label: i for i, label in enumerate(all_labels)}

# 3. COMBINE LINKS #
# institution_type → instrument
links0 = glcf0.copy()
links0['source'] = links0['institution_type0'].map(label_to_index)
links0['target'] = links0['instrument_combined'].map(label_to_index)

# instrument → region
links1 = glcf1.copy()
links1['source'] = links1['instrument_combined'].map(label_to_index)
links1['target'] = links1['region_destination_cpi0'].map(label_to_index)

# region → use
links2 = glcf2.copy()
links2['source'] = links2['region_destination_cpi0'].map(label_to_index)
links2['target'] = links2['use_cpi'].map(label_to_index)

# Final link dataframe
all_links = pd.concat([
    links0[['source', 'target', 'total']],
    links1[['source', 'target', 'total']],
    links2[['source', 'target', 'total']]
], ignore_index=True)


In [ ]:
# 4. COLOR ASSIGNMENT # 
highlight_label = "International: Debt"
highlight_color_1 = "#D76C82"  # burgundy with opacity
highlight_color_2 = "rgba(255,165,0,0.8)"   # orange for "Rest of the World"
default_link_color = "rgba(192,192,192,0.3)"  # light gray
default_node_color = "lightgray"

# Assign node colors
node_colors = []
for label in all_labels:
    if label == "International: Debt":
        node_colors.append(highlight_color_1)
    elif label == "Public_multilateral":
        node_colors.append(highlight_color_1)
    elif label == "Public_Bilat":
        node_colors.append(highlight_color_1)
    elif label == "International: Equity":
        node_colors.append(highlight_color_1)
    elif label == "International: Grant":
        node_colors.append(highlight_color_1)
    elif label == "Rest of the World":
        node_colors.append(highlight_color_2)
    else:
        node_colors.append(default_node_color)

# Assign link colors 

link_colors = []
for _, row in all_links.iterrows():
    src = int(row['source'])
    tgt = int(row['target'])
    src_label = all_labels[src]
    tgt_label = all_labels[tgt]

    if src_label == "International: Debt" or tgt_label == "International: Debt":
        link_colors.append(highlight_color_1)
    elif src_label == "International: Equity" or tgt_label == "International: Equity":
        link_colors.append(highlight_color_1)
    elif src_label == "International: Grant" or tgt_label == "International: Grant":
        link_colors.append(highlight_color_1)
    elif src_label == "Rest of the World" or tgt_label == "Rest of the World":
        link_colors.append(highlight_color_2)
    else:
        link_colors.append(default_link_color)

In [ ]:
# Create Sankey figure
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=10,
        thickness=70,
        label=[""] * len(all_labels),  # removes text
        color=node_colors
    ),
    link=dict(
        source=all_links['source'].astype(int),
        target=all_links['target'].astype(int),
        value=all_links['total'],
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sector → Instrument → Region → Use",
    font_size=13,
    width=1000,
    height=600
)

fig.show()

# --- Fig. S1 | financeflows -----------------------------------------------
# Thick node bars, no node text: box labels, column headers and dollar values are added by hand.
# Export needs `pip install "kaleido<1"` for plotly 6.
import os
_OUT = os.path.join('4_figures', 'supplementary')
os.makedirs(_OUT, exist_ok=True)
fig.write_image(os.path.join(_OUT, 'financeflows.png'), scale=3)
print('wrote', os.path.join(_OUT, 'financeflows.png'))


In [ ]:
glcf_filtered[glcf_filtered['institution_type0'] == 'Public_multilateral']['value_USDm'].sum() /2

In [ ]:
glcf_filtered[glcf_filtered['institution_type0'] == 'Public_Bilat']['value_USDm'].sum() / 2

In [ ]:
glcf[glcf['instrument_cpi']=='Low-cost project debt']['value_USDm'].sum()/2